In [11]:
import numpy as np

In [1]:
import pandas as pd

In [5]:
#Data Processing
patients = pd.read_csv("patients.csv")
diagnoses = pd.read_csv("diagnoses.csv")
outcomes = pd.read_csv("outcomes.csv")
labs = pd.read_csv("labs.csv")

In [8]:
patients = patients.merge(diagnoses , on =  "DiagnosisID")
patients = patients.merge(outcomes , on = "OutcomeID")

In [9]:
patients['AdmissionDate'] = pd.to_datetime(patients['AdmissionDate'])
patients["DischargeDate"] = pd.to_datetime(patients["DischargeDate"])
patients["LengthofStay"] = (patients["DischargeDate"]- patients['AdmissionDate']).dt.days

In [10]:
patients['OutcomeEncoded'] = patients['OutcomeName'].map({'Recovered' : 0, 'Complicated' : 1, 'Deceased' : 1})

In [12]:
patients["HighRisk"] = np.where((patients["Age"]> 65)| (patients["OutcomeName"].isin(["Complicated", "Deceased"])),1,0)

In [16]:
abnormal_conditions = {
    "Blood Sugar" : lambda x : x > 120,
    "Cholestrol" : lambda x : x > 200,
    "Hemoglobin" : lambda x : x < 13
    }

def count_abnormal_labs(patient_id):
    patient_labs = labs[labs['PatientID'] == patient_id]
    count = 0
    for test_name, condition  in abnormal_conditions.items():
        test_results = patient_labs[patient_labs["TestName"] == test_name]
        count += test_results["Result"].apply(condition).sum()
    return count
patients['AbnormalLabCount'] = patients['PatientID'].apply(count_abnormal_labs)

Model Training

In [19]:
features = patients[['Age','LengthofStay','TreatmentCost','AbnormalLabCount']]
target = patients['OutcomeEncoded']



In [20]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size = 0.3, random_state= 42)

In [21]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=500)
model.fit(x_train, y_train)

LogisticRegression(max_iter=500)

In [22]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(x_test)
print("classification report : ")
print(classification_report(y_test, y_pred))

classification report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        14
           1       0.77      1.00      0.87        46

    accuracy                           0.77        60
   macro avg       0.38      0.50      0.43        60
weighted avg       0.59      0.77      0.67        60



C:\Users\pc\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\pc\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\pc\AppData\Roaming\Python\Python311\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [23]:
import joblib
joblib.dump(model, 'riskmodel.ipynb')

['riskmodel.ipynb']